[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/klein_quadric/klein_quadric.ipynb)

# Four Lines and the Klein Quadric

This example finds the lines that meet four given lines in space. Three lines in general position have a whole family of lines meeting all three, sweeping out a hyperboloid; a fourth line crosses that hyperboloid in two points, and through each passes one line meeting all four. When the fourth line misses the hyperboloid, the two lines are still there, as a complex-conjugate pair.

A line in space is a bivector of projective geometric algebra, but most bivectors are not lines: a bivector is a line when its wedge with itself vanishes, and two lines meet when their wedge vanishes. Both are one form, `Bivector ^ Bivector`, a quadratic form on the six-dimensional space of bivectors whose zeros are the lines: the Klein quadric. Meeting four lines is four linear conditions, which leave a pencil of bivectors; along the pencil the Klein form is a quadratic, and its two zeros are the two transversals. The algebra runs on complex coefficients, so that both are there in every case.

In Plücker coordinates a line reads as six numbers on the Klein quadric in five-dimensional projective space, and in the Schubert calculus of the Grassmannian of lines the count of two reads as the intersection number of four Schubert cycles.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from examples.animation import save_figure, show_animation
from examples.math.klein_quadric import render, scenarios
from examples.math.klein_quadric.core import Bivector, HORIZON, KLEIN, ORIGIN, Plane, Point, Quadric, SIGNS, Scalar, mv, spread

np.set_printoptions(precision=4, suppress=True)

## 1. The Klein form

The Klein form takes two bivectors to their wedge, a pseudoscalar. A meet of two planes is a line and the form vanishes on it twice; a sum of two skew lines is not a line, and the form does not. Two lines that cross pair to zero, and two that pass each other do not.

In [ ]:
# Two lines along z, through the origin and through (1, 0, 0), each the meet of two planes; and a line
# along y through (1, 0, 1), passing the first.
along_z = mv.x ^ mv.y                                                          # [] Bivector
beside = (mv.x - mv.w) ^ mv.y                                                  # [] Bivector
above = (mv.x - mv.w) ^ (mv.z - mv.w)                                          # [] Bivector

In [ ]:
print("a line with itself:                     ", KLEIN(along_z, along_z).dual().to_array())
print("a sum of two skew lines with itself:    ", KLEIN(along_z + above, along_z + above).dual().to_array())
print("two parallel lines, meeting at infinity:", KLEIN(along_z, beside).dual().to_array())
print("two skew lines:                         ", KLEIN(along_z, above).dual().to_array())

## 2. Four lines, two transversals

Each line, weighted by its Klein pairing with a bivector, is a map whose zeros are the bivectors that meet it; summed over the four lines, the map is zero only on the bivectors that meet all four. They form a pencil, the two singular vectors of zero singular value. Along the pencil the Klein form is a quadratic, and its two roots are the two transversals. Three lines are fixed through the waist of the hyperboloid x² + y² = 1 + z², and a steep fourth line is placed outside the waist and through it.

In [ ]:
# Where the three lines cross the hyperboloid's waist, by angle about z, and the fourth line's offsets
# along x: outside the waist, and through it.
waist_angles = np.array([0.3, 2.3, 4.4])
offsets = np.array([1.4, 0.0])

In [ ]:
def roots(b: Scalar, c: Scalar) -> Scalar:
    """The two roots of the quadratic `1 + 2 * b * t + c * t * t` along the combinations of two ends,
    as the weight of the first end against unit weight on the second, one root per sign on the last
    axis: `first * (sign * root - b) + second`, with `root * root == b * b - c`. A quadratic whose
    coefficients share a factor, such as the handedness of a signed volume, is divided by its first
    coefficient to read as this one."""
    root = (b * b - c).square_root()                                           # [...] Scalar
    return root[..., None] * SIGNS - b[..., None]                              # [..., 2] Scalar


def transversals(lines: Bivector) -> Bivector:
    """The two lines that meet all four lines on the last axis, at unit norm, on the last axis."""
    # Each line, weighted by the Klein form between it and a bivector: zero exactly on the bivectors
    # that meet all four.
    misses = (lines * KLEIN(lines).dual()).sum(axis=-1)                        # [...] Bivector <- Bivector
    # Those bivectors form a pencil, spanned by the right singular vectors of zero singular value.
    _, _, right = misses.svd()
    pencil = right[..., -2:]                                                   # [..., 2] Bivector
    # Along the pencil the Klein form is a quadratic; its two zeros are the lines of the pencil.
    first, second = pencil[..., 0], pencil[..., 1]                             # [...] Bivector
    # The Klein form of two axes is a signed volume; its ratios are ordinary numbers.
    leading = KLEIN(first, first).dual()                                       # [...] Scalar
    weights = roots(KLEIN(first, second).dual() / leading, KLEIN(second, second).dual() / leading)   # [..., 2] Scalar
    return (first[..., None] * weights + second[..., None]).normalized()       # [..., 2] Bivector


three = scenarios.waist_lines(waist_angles)                                     # [3] Bivector
fourth = scenarios.steep(offsets)                                              # [offsets] Bivector
lines = stack([three[0].broadcast_to(fourth.shape), three[1].broadcast_to(fourth.shape),
               three[2].broadcast_to(fourth.shape), fourth], axis=-1)          # [offsets, 4] Bivector
across = transversals(lines)                                                   # [offsets, 2] Bivector

In [ ]:
print("transversals real, outside the waist and through it:", render.real(across.kernel))
print("largest Klein pairing of a transversal with any of the four:", np.abs((lines[..., :, None] ^ across[..., None, :]).kernel).max())

## 3. The same count, seen in space

The lines that meet three lines sweep out a quadric. As a map from points to planes it pairs to zero with its points: the plane joining a point to the first line meets the third line in a point, and the plane joining that point to the second line holds the first point exactly when a line through it meets all three. The fourth line crosses that quadric at two points, by the same quadratic as before, now along the fourth line, and through each crossing runs one of the transversals. In the animation the fourth line, orange, swings through the waist and back: outside, the two crossings and the red transversals are real; inside, both turn complex together, and the red lines vanish from the real picture.

In [ ]:
# The lines drawn of each family; the frames of the swing and how far it reaches either way; the frame
# shown as a still; and the duration of a frame in milliseconds.
n_rulings = 36
n_frames = 96
swing = 1.5
still = 8
frame_duration = 70

In [ ]:
def ruled_quadric(first: Bivector, second: Bivector, third: Bivector) -> Quadric:
    """The quadric swept by the lines that meet three lines, as a map from points to planes that
    pairs to zero with the points on it. The plane joining a point to the first line meets the third
    line in a point; the plane joining that point to the second line holds the first point exactly
    when a line through it meets all three."""
    return ((Point & first) ^ third) & second                                  # [...] Plane <- Point


def crossings(surface: Quadric, line: Bivector) -> Point:
    """The two points where a line crosses a quadric, at unit weight, on the last axis."""
    # The line's point nearest the origin and its point at infinity span it.
    span = stack([(ORIGIN | line) ^ line, line ^ HORIZON], axis=-1)            # [..., 2] Point
    first, second = span[..., 0], span[..., 1]                                 # [...] Point
    # The quadric's map need not be symmetric: the cross term is the mean of both orders.
    cross = ((surface(first) & second) + (surface(second) & first)) / 2        # [...] Scalar
    leading = surface(first) & first                                           # [...] Scalar
    weights = roots(cross / leading, (surface(second) & second) / leading)     # [..., 2] Scalar
    crossing = first[..., None] * weights + second[..., None]                  # [..., 2] Point
    return crossing / (mv.w & crossing)                                        # [..., 2] Point


def through(points: Point, second: Bivector, third: Bivector) -> Bivector:
    """The line through each point that meets two lines: the meet of the planes joining the point to
    each line."""
    return (points & second) ^ (points & third)                                # [...] Bivector


swung = scenarios.steep(swing * np.cos(np.linspace(0.0, 2 * np.pi, n_frames, endpoint=False)))   # [frames] Bivector
swung_lines = stack([three[0].broadcast_to(swung.shape), three[1].broadcast_to(swung.shape),
                     three[2].broadcast_to(swung.shape), swung], axis=-1)       # [frames, 4] Bivector
swung_across = transversals(swung_lines)                                       # [frames, 2] Bivector
surface = ruled_quadric(three[0], three[1], three[2])                          # [] Plane <- Point
crossing = crossings(surface, swung)                                           # [frames, 2] Point
# The lines meeting the three, through points spread over the first; and the lines meeting three of
# those, the family of the three.
sweep = through(spread(three[0], n_rulings), three[1], three[2])               # [rulings] Bivector
family = through(spread(sweep[0], n_rulings), sweep[n_rulings // 3], sweep[2 * n_rulings // 3])   # [rulings] Bivector
rulings = stack([sweep, family])                                               # [families, rulings] Bivector
scenes = [(swung_lines[frame], rulings, swung_across[frame], crossing[frame]) for frame in range(n_frames)]
save_figure(render.draw(*scenes[still], azimuth=render.AZIMUTH), "klein_transversals")
show_animation(render.animate(scenes), "klein_swing", frame_duration)

In [ ]:
# Each crossing's distance from the nearer transversal: zero, real or complex.
off = np.abs((crossing[..., :, None] & swung_across[..., None, :]).kernel).max(axis=-1).min(axis=-1)   # [frames, 2]
print("largest distance of a crossing from its transversal:", off.max())
print("frames with real transversals:", render.real(swung_across.kernel).all(axis=-1).sum(), "of", n_frames)

In [ ]:
# checks
# The Klein form vanishes on lines and on lines that meet; the transversals are lines and meet all four;
# the crossings lie on the quadric and on the transversals; both families lie on the quadric; outside the
# waist both transversals are real, through it both are complex.
np.testing.assert_allclose(KLEIN(along_z, along_z).dual().to_array(), 0.0, atol=1e-12)
np.testing.assert_allclose(KLEIN(along_z, beside).dual().to_array(), 0.0, atol=1e-12)
np.testing.assert_allclose((swung_across ^ swung_across).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((swung_lines[..., :, None] ^ swung_across[..., None, :]).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((surface(crossing) & crossing).kernel, 0.0, atol=1e-10)
np.testing.assert_allclose(off, 0.0, atol=1e-10)
ends = spread(rulings, 3)                                                      # [families, rulings, 3] Point
np.testing.assert_allclose((surface(ends) & ends).kernel, 0.0, atol=1e-10)
assert render.real(across[0].kernel).all() and not render.real(across[1].kernel).any()